# Людина в циклі: переддіяльні бар’єри, ранжування ризиків та журнал аудиту

README цього уроку представляє Людину в циклі з коротким фрагментом, який запитує у користувача `APPROVE` або `REJECT` після того, як агент уже створив відповідь. Такий підхід є гарним початком, але виробничі реалізації HITL зазвичай потребують ще три додаткові елементи:

1. **Переддіяльний бар’єр**, який запускається **до** того, як агент виконає ризиковий крок, щоб контроль витрат, незворотності та затримок був під контролем.
2. **Ранжування ризиків**, щоб дії з низьким ризиком виконувалися автоматично, дії зі середнім ризиком схвалювалися пакетно, а лише дії з високим ризиком блокувалися людиною.
3. **Журнал аудиту та цикл ревізій**, щоб кожне рішення бар’єру фіксувалося у форматі JSONL, а відхилення повторно запускало агента з структурованою причиною замість простого виводу `Revising...`.

Ця записна книжка будує кожен з цих елементів на основі тих самих примітивів, що й у `06-system-message-framework.ipynb`. Вона виконується наскрізно у режимі `DEMO_MODE = True` (не потрібен інтерактивний ввід) або з реальними підказками `input()`, коли `DEMO_MODE = False`. Примітка: у DEMO_MODE повторна спроба третьої мети запрограмована, тому механіка циклу видима наскрізно. Справжнє повторне класифікування, кероване ревізією, вимагає `DEMO_MODE = False` та оператора.

**Поза межами цієї теми (розглядається в інших уроках):** автентифікація та контроль доступу (Урок 06 README загроза 2), проміжне ПЗ для виклику інструментів (Урок 14 глибокий розбір MAF), шаблони дебатів кількох агентів.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Шаблон 1: Вузол перед дією

Фрагмент HITL у README спочатку викликає агента, а потім просить користувача схвалити результат. Це потік **після дії**. Агент вже виконався, тому вартість виклику LLM вже оплачена, і будь-який побічний ефект (надісланий імейл, запис у базі даних, опублікований коментар) вже відбувся.

Потік **перед дією** вставляє вузол перед тим, як агент виконає ризиковий крок. Агент пропонує дію, вузол вирішує, чи виконувати її, і лише за схвалення побічний ефект відбувається.

| Аспект | Схвалення після дії (фрагмент README) | Вузол перед дією (цей ноутбук) |
|---|---|---|
| Коли відбувається схвалення? | Після того, як агент видав результат | Перед виконанням будь-якого побічного ефекту |
| Вартість LLM при відхиленні | Вже оплачена | Оплачена лише за пропозицію, а не за дію |
| Незворотні побічні ефекти | Можливі (дія вже відбулась) | Запобігаються |
| Чіткість аудиту | Схвалення у вигляді друку | Схвалення у вигляді JSON-запису з таймстемпом, дією, причиною |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: Розподіл за рівнями ризику

Не кожна дія потребує схвалення людиною. Перегляд інформації через публічний API з правами лише для читання має інші наслідки, ніж надсилання електронного листа клієнту. Обробка їх однаково марнує увагу оператора і сповільнює агента.

Проста модель з 3 рівнями:

| Рівень | Приклади | Потік схвалення |
|---|---|---|
| `низький` (лише читання) | Пошук у базі знань, перегляд варіантів рейсів, отримання публічної веб-сторінки | Автоматичне виконання, запис для аудиту |
| `середній` (недорога зміна) | Кешування результату, інкремент лічильника, планування нагадування | Автоматичне виконання, але з подальшим щоденним переглядом |
| `високий` (зовнішній або незворотний) | Надсилання листа, зняття коштів з картки, публікація у публічному каналі | Блокування до схвалення людиною |

Це один із варіантів розподілу. Промислові системи часто використовують більш деталізовані рівні (наприклад, рівні дозволів AWS IAM, рольові рівні доступу). Версія з 3 рівнями нижче є найменш корисною для агента, який поєднує дії лише для читання з такими, що мають побічні ефекти.

Класифікатор нижче використовує евристику за ключовими словами, щоб демонстрація залишалася детермінованою та недорогою. У промисловій системі ви замінили б це на навчений класифікатор або політичний движок.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Шаблон 3: Журнал аудиту та цикл ревізії

`print("Response approved.")` не є журналом аудиту. Для довіри кожне рішення про допуск має записуватися як структурована подія, яку пізніше можна запитувати, відтворювати або додавати до перегляду інцидентів.

Дві складові:

1. **Додатковий лише JSONL.** Один рядок на рішення з часовою позначкою, дією, рівнем, рішенням, причиною. Легко шукати за допомогою grep, легко пізніше відправляти у справжнє сховище журналів.
2. **Цикл ревізії при відхиленні.** Коли ворота повертають `deny`, агент повторно вводить собі підказку з причиною відмови в контексті, щоб наступна пропозиція могла уникнути проблеми.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Додаткові ресурси

Декілька інших публічних проектів реалізують варіації цих HITL патернів. Порівняйте підходи, щоб знайти те, що підходить вашому стеку:

- **LangChain** обгортки інструментів human-in-the-loop ([документація](https://python.langchain.com/docs/integrations/tools/human_tools)): обгортки інструментів, які зупиняють виконання для отримання людського введення.
- **AutoGen** `UserProxyAgent` ([документація v0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ змінив цю структуру): використовує роль агента спеціально для представлення людини у багатокористувацьких бесідах.
- **Semantic Kernel** фільтри функцій ([документація](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): проміжне ПЗ, що виконується навколо кожного виклику функції, підходить для керування логікою доступу.

Кожен проект по-різному обробляє три підпатерни: LangChain обгортає їх як інструменти, AutoGen використовує роль агента, Semantic Kernel використовує фільтри проміжного ПЗ. Прочитайте одну чи дві реалізації цілком перед тим, як обирати дизайн для власного агента.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
